# SUGAR · Browser edition
Search public social posts, translate with ARC or OpenAI, and create downloadable spreadsheets, maps, and reports.

**Run all cells, then use the form below.** The first package load may take a minute. No API requests run until you click a workflow button. Try **Load demo data** first; it makes no social/LLM API calls.

- This version runs Python locally in your browser. It does not use desktop paths, subprocesses, or a local Python installation.
- Upload CSV/XLSX or saved Nitter HTML using the form. Downloads are also written to `sugar-output` in JupyterLite's browser filesystem.
- Live X, Bluesky, Mastodon, ARC/OpenAI, and geocoding requests depend on each service's CORS and authentication rules. VPN access does not override CORS. X may require a server connector. No proxy is bundled.
- Keys/passwords are runtime inputs, never embedded in this notebook or exports. Widget fields are cleared after a run. Do not save widget state containing credentials.
- Public location inference is optional, broad, and uncertain; inferred locations are not verified geotags. Geocoding is separately opt-in.
- The missing desktop `sugar_analysis.py` is replaced by a descriptive report (counts, dates, languages, sources and sample posts): HTML, Word, and a print-to-PDF version. It is not a reproduction of that missing module.
- This is a standalone notebook. The existing Ask ARC HTML launcher remains unchanged and targets its original simple notebook.


In [ ]:
# Browser-compatible packages; no subprocess, desktop pip, or SSL overrides.
import piplite
await piplite.install(["pandas", "beautifulsoup4", "langdetect", "python-dateutil", "openpyxl", "folium", "ipywidgets", "python-docx"])
print("SUGAR browser packages are ready.")


In [ ]:
from __future__ import annotations
import asyncio
import base64
import csv
import html as html_lib
import json
import os
import re
import types
from dataclasses import asdict, dataclass, fields
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional
from urllib.parse import urlencode, urljoin, urlparse

import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
from bs4 import BeautifulSoup
from langdetect import detect, LangDetectException, DetectorFactory
from dateutil import parser as dateparser
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from IPython.display import display, HTML

DetectorFactory.seed = 0
OpenAI = Any  # Only a compatibility annotation; the desktop SDK is not used.
tqdm = None
X_API_BASE_URL = 'https://api.x.com/2'
BLUESKY_API_BASE_URL = 'https://public.api.bsky.app'
BLUESKY_PDS_URL = 'https://bsky.social'
BLUESKY_SERVICE_PROXY = 'did:web:api.bsky.app#bsky_appview'
ARC_BASE_URL = 'https://llm-api.arc.vt.edu/api/v1'
LLM_PROVIDER = 'arc'
LLM_MODEL = 'gpt-oss-120b'
LLM_API_KEY = ''
LLM_BASE_URL = ARC_BASE_URL
INFER_LOCATIONS = False
GEOCODE_LOCATIONS = False
GEOCODE_ENDPOINT = 'https://nominatim.openstreetmap.org/search'
CREATE_MAP = False
DEFAULT_SEARCH_TERMS = ['Democracy']
REQUEST_DELAY_SECONDS = 3.0
TRANSLATION_DELAY_SECONDS = 0.5
GEOCODE_DELAY_SECONDS = 1.2
MIN_LOCATION_CONFIDENCE = 0.2
RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path('sugar-output')
OUTPUT_DIR.mkdir(exist_ok=True)
GEOCODE_CACHE_FILE = str(OUTPUT_DIR / 'geocode_cache.json')
MAP_FILE = str(OUTPUT_DIR / f'social_search_map_{RUN_TIMESTAMP}.html')
MAP_BASE_TILE_NAME = 'CartoDB Positron'
MAP_BASE_TILE_URL = 'https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png'
MAP_BASE_TILE_ATTRIBUTION = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'
ADD_ALTERNATE_BASEMAPS = True
NETWORK_FAILURES = []

class BrowserRequestError(RuntimeError):
    pass

class BrowserResponse:
    def __init__(self, status, text, url, headers=None):
        self.status_code, self.text, self.url = status, text, url
        self.headers = headers or {}
    def json(self):
        return json.loads(self.text)
    def raise_for_status(self):
        if not 200 <= self.status_code < 300:
            raise BrowserRequestError(f'HTTP {self.status_code} from {urlparse(self.url).netloc}')

class BrowserSession:
    """A small requests-style adapter backed by browser fetch, never sockets."""
    async def request(self, method, url, *, params=None, headers=None, json=None, timeout=60):
        from pyodide.http import pyfetch
        if urlparse(url).scheme != 'https':
            raise ValueError('API addresses must use HTTPS.')
        if params:
            url += ('&' if '?' in url else '?') + urlencode(params, doseq=True)
        options = {'method': method, 'headers': dict(headers or {}), 'credentials': 'omit'}
        if json is not None:
            options['headers']['Content-Type'] = 'application/json'
            options['body'] = globals()['json'].dumps(json)
        try:
            async def read_response():
                response = await pyfetch(url, **options)
                return BrowserResponse(response.status, await response.string(), url,
                                       {'Content-Type': response.headers.get('content-type', '')})
            return await asyncio.wait_for(read_response(), timeout=timeout)
        except asyncio.CancelledError:
            raise
        except Exception as exc:
            host = urlparse(url).netloc
            message = (f'Browser request to {host} failed. Check internet/VPN access and the API’s '
                       'CORS support. A VPN does not bypass CORS. This source may require a server-side '
                       'connector; you can still upload previously collected data.')
            NETWORK_FAILURES.append(message)
            raise BrowserRequestError(message) from exc
    async def get(self, url, **kwargs):
        return await self.request('GET', url, **kwargs)
    async def post(self, url, **kwargs):
        return await self.request('POST', url, **kwargs)

requests = types.SimpleNamespace(Session=BrowserSession, RequestException=BrowserRequestError,
                                exceptions=types.SimpleNamespace(RequestException=BrowserRequestError))

def create_requests_session():
    return BrowserSession()

def create_llm_client():
    if not LLM_API_KEY:
        raise ValueError('Enter an LLM API key to translate or infer locations.')
    return {'provider': LLM_PROVIDER, 'model': LLM_MODEL, 'key': LLM_API_KEY, 'base': LLM_BASE_URL}

async def generate_llm_text(client, model, prompt):
    session = BrowserSession()
    if client['provider'] == 'arc':
        response = await session.post(client['base'].rstrip('/') + '/chat/completions',
            headers={'Authorization': 'Bearer ' + client['key']},
            json={'model': model, 'messages': [{'role':'user','content':prompt}], 'max_tokens':8000}, timeout=120)
    else:
        response = await session.post('https://api.openai.com/v1/responses',
            headers={'Authorization': 'Bearer ' + client['key']},
            json={'model': model, 'input': prompt}, timeout=120)
    response.raise_for_status()
    data = response.json()
    if client['provider'] == 'arc':
        return str(data['choices'][0]['message'].get('content') or '').strip()
    text = '\n'.join(part.get('text','') for item in data.get('output',[])
                     for part in item.get('content',[]) if part.get('type') == 'output_text')
    if not text:
        raise BrowserRequestError('The model did not return text. Check its response or model ID.')
    return text.strip()

async def geocode_location(location_name, cache, geolocator=None, delay_seconds=1.2):
    """Optional, sequential Nominatim lookup with a local cache and browser Referer."""
    empty = {'latitude':'', 'longitude':'', 'geocode_display_name':''}
    if not GEOCODE_LOCATIONS or not location_name.strip():
        return empty
    if location_name in cache:
        return cache[location_name]
    await asyncio.sleep(max(1.2, delay_seconds))
    try:
        response = await BrowserSession().get(GEOCODE_ENDPOINT,
            params={'q':location_name, 'format':'jsonv2', 'limit':1}, timeout=25)
        response.raise_for_status()
        places = response.json()
        data = ({'latitude':str(places[0]['lat']), 'longitude':str(places[0]['lon']),
                 'geocode_display_name':str(places[0].get('display_name',''))} if places else empty)
        cache[location_name] = data
        save_json_cache(GEOCODE_CACHE_FILE, cache)
        return data
    except BrowserRequestError as exc:
        print(f'Geocoding unavailable: {exc}')
        return empty

def safe_http_url(value):
    value = str(value or '').strip()
    return value if urlparse(value).scheme in {'https','http'} else ''

@dataclass
class PostRecord:
    platform: str
    query: str
    source_mode: str
    source_host: str
    source_url: str
    scraped_from: str
    post_url: str
    x_url: str
    tweet_id: str
    date_raw: str
    date_iso: str
    username: str
    display_name: str
    detected_language: str
    original_text: str
    translated_en: str
    raw_stats: str
    is_retweet: bool
    inferred_location: str
    location_confidence: float
    location_source: str
    location_reason: str
    latitude: str
    longitude: str
    geocode_display_name: str

def clean_handle(handle: str) -> str:
    handle = handle.strip()
    if handle.startswith('@'):
        handle = handle[1:]
    return handle

def normalize_whitespace(text: str) -> str:
    return re.sub('\\s+', ' ', text or '').strip()

def clean_cell(value, prevent_formula_injection: bool=True):
    """
    Make social media text safer and cleaner for CSV/Excel output.
    Removes internal line breaks, normalizes spacing, and prevents
    spreadsheet formula injection.
    """
    if value is None:
        return ''
    value = str(value)
    value = value.replace('\r', ' ')
    value = value.replace('\n', ' ')
    value = value.replace('\t', ' ')
    value = normalize_whitespace(value)
    if prevent_formula_injection and value.startswith(('=', '+', '-', '@')):
        value = "'" + value
    return value

def detect_language_safe(text: str) -> str:
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'
    except Exception:
        return 'unknown'

def parse_nitter_date(date_text: str) -> str:
    if not date_text:
        return ''
    cleaned = date_text.replace('·', ' ').strip()
    try:
        dt = dateparser.parse(cleaned, fuzzy=True)
        if dt:
            return dt.isoformat()
    except Exception:
        return ''
    return ''

def build_queries(terms: List[str], handles: Optional[List[str]]=None) -> List[str]:
    if handles:
        queries = []
        for handle in handles:
            h = clean_handle(handle)
            queries.append(f'from:{h}')
        return queries
    return [term.strip() for term in terms if term.strip()]

async def create_bluesky_access_token(session: requests.Session, identifier: str, app_password: str) -> str:
    """Create an in-memory Bluesky session and return its short-lived JWT."""
    endpoint = f"{BLUESKY_PDS_URL.rstrip('/')}/xrpc/com.atproto.server.createSession"
    response = await session.post(endpoint, json={'identifier': identifier, 'password': app_password}, headers={'Accept': 'application/json'}, timeout=60)
    try:
        payload = response.json()
    except ValueError as exc:
        raise RuntimeError(f'Bluesky login returned a non-JSON response ({response.status_code}).') from exc
    if response.status_code in {400, 401}:
        message = payload.get('message') or payload.get('error') or 'login rejected'
        raise RuntimeError(f'Bluesky rejected the handle or app password: {message}')
    try:
        response.raise_for_status()
    except requests.RequestException as exc:
        message = payload.get('message') or payload.get('error') or 'unknown error'
        raise RuntimeError(f'Bluesky login failed ({response.status_code}): {message}') from exc
    token = str(payload.get('accessJwt', '')).strip()
    if not token:
        raise RuntimeError('Bluesky login succeeded but returned no access token.')
    return token

def x_api_time(value: Optional[str], end_of_day: bool=False) -> Optional[str]:
    """Convert a date setting to the RFC 3339 form expected by X."""
    if not value:
        return None
    value = value.strip()
    if re.fullmatch('\\d{4}-\\d{2}-\\d{2}', value):
        return value + ('T23:59:59Z' if end_of_day else 'T00:00:00Z')
    return value

def build_x_query(query: str, include_retweets: bool, languages: Optional[List[str]]=None) -> str:
    """Add selected language and retweet filters to an X query."""
    query = normalize_whitespace(query)
    if languages and (not re.search('(?:^|\\s)lang:', query, flags=re.I)):
        language_filter = ' OR '.join((f'lang:{code}' for code in languages))
        query = f'{query} ({language_filter})'
    if not include_retweets and (not re.search('(?:^|\\s)-?is:retweet(?:\\s|$)', query)):
        query = f'{query} -is:retweet'
    return query

async def fetch_x_api_page(session: requests.Session, bearer_token: str, query: str, max_results: int, search_mode: str='recent', since: Optional[str]=None, until: Optional[str]=None, next_token: Optional[str]=None) -> tuple[List[Dict[str, Any]], Optional[str], str]:
    """Fetch and normalize one page from X API v2 recent or archive search."""
    if search_mode not in {'recent', 'all'}:
        raise ValueError(f'Unsupported X search mode: {search_mode}')
    endpoint = f"{X_API_BASE_URL.rstrip('/')}/tweets/search/{search_mode}"
    endpoint_limit = 500 if search_mode == 'all' else 100
    params: Dict[str, Any] = {'query': query, 'max_results': max(10, min(endpoint_limit, max_results)), 'tweet.fields': 'id,text,author_id,created_at,lang,public_metrics,referenced_tweets', 'expansions': 'author_id', 'user.fields': 'id,name,username,location'}
    if since:
        params['start_time'] = x_api_time(since)
    if until:
        params['end_time'] = x_api_time(until, end_of_day=False)
    if next_token:
        params['next_token'] = next_token
    response = await session.get(endpoint, params=params, headers={'Authorization': f'Bearer {bearer_token}'}, timeout=60)
    if response.status_code == 401:
        raise RuntimeError('X rejected the Bearer Token (401 Unauthorized).')
    if response.status_code == 402:
        raise RuntimeError('X reported that API credits or billing are required (402).')
    if response.status_code == 403:
        raise RuntimeError(f'X denied access to {search_mode} search for this app (403 Forbidden). Full archive requires eligible Pay-per-use or Enterprise access.')
    if response.status_code == 429:
        raise RuntimeError('X API rate limit reached (429). Try again later.')
    try:
        response.raise_for_status()
        payload = response.json()
    except requests.RequestException as exc:
        detail = response.text[:500] if response is not None else ''
        raise RuntimeError(f'X API request failed: {exc}. {detail}') from exc
    except ValueError as exc:
        raise RuntimeError('X API returned a non-JSON response.') from exc
    errors = payload.get('errors') or []
    if errors and (not payload.get('data')):
        raise RuntimeError(f'X API returned errors: {json.dumps(errors)}')
    users = {str(user.get('id', '')): user for user in payload.get('includes', {}).get('users', [])}
    posts: List[Dict[str, Any]] = []
    for item in payload.get('data', []) or []:
        author = users.get(str(item.get('author_id', '')), {})
        username = str(author.get('username', '')).strip()
        tweet_id = str(item.get('id', '')).strip()
        referenced = item.get('referenced_tweets') or []
        is_retweet = any((ref.get('type') == 'retweeted' for ref in referenced))
        created_at = str(item.get('created_at', '')).strip()
        public_metrics = item.get('public_metrics') or {}
        x_url = f'https://x.com/{username}/status/{tweet_id}' if username else f'https://x.com/i/web/status/{tweet_id}'
        posts.append({'platform': 'x', 'query': query, 'scraped_from': response.url, 'source_url': response.url, 'post_url': x_url, 'x_url': x_url, 'tweet_id': tweet_id, 'date_raw': created_at, 'date_iso': created_at, 'username': username, 'display_name': str(author.get('name', '')).strip(), 'original_text': str(item.get('text', '')), 'raw_stats': json.dumps(public_metrics, sort_keys=True), 'is_retweet': is_retweet})
    pagination_token = payload.get('meta', {}).get('next_token')
    return (posts, pagination_token, response.url)

async def fetch_bluesky_page(session: requests.Session, query: str, max_results: int, since: Optional[str]=None, until: Optional[str]=None, cursor: Optional[str]=None, access_jwt: str='') -> tuple[List[Dict[str, Any]], Optional[str], str]:
    """Fetch one Bluesky page publicly or through an authenticated PDS proxy."""
    if access_jwt:
        endpoint = f"{BLUESKY_PDS_URL.rstrip('/')}/xrpc/app.bsky.feed.searchPosts"
        headers = {'Accept': 'application/json', 'Authorization': f'Bearer {access_jwt}', 'atproto-proxy': BLUESKY_SERVICE_PROXY}
    else:
        endpoint = f"{BLUESKY_API_BASE_URL.rstrip('/')}/xrpc/app.bsky.feed.searchPosts"
        headers = {'Accept': 'application/json'}
    params: Dict[str, Any] = {'q': query, 'sort': 'latest', 'limit': max(1, min(100, max_results))}
    if since:
        params['since'] = since
    if until:
        params['until'] = until
    if cursor:
        params['cursor'] = cursor
    response = await session.get(endpoint, params=params, headers=headers, timeout=60)
    try:
        response.raise_for_status()
        payload = response.json()
    except requests.RequestException as exc:
        content_type = response.headers.get('Content-Type', '').lower()
        if response.status_code == 403 and 'html' in content_type:
            raise RuntimeError('the public Bluesky API was blocked by a network filter (HTML 403). Run the script again and enter your Bluesky handle plus an app password, or use a network/VPN that permits *.bsky.app.') from exc
        detail = normalize_whitespace(response.text[:300])
        raise RuntimeError(f'Bluesky API request failed ({response.status_code}): {detail}') from exc
    except ValueError as exc:
        raise RuntimeError('Bluesky returned a non-JSON response.') from exc
    posts: List[Dict[str, Any]] = []
    for item in payload.get('posts', []) or []:
        author = item.get('author') or {}
        record = item.get('record') or {}
        uri = str(item.get('uri', ''))
        rkey = uri.rsplit('/', 1)[-1] if '/' in uri else ''
        handle = str(author.get('handle', '')).strip()
        post_url = f'https://bsky.app/profile/{handle}/post/{rkey}' if handle and rkey else ''
        metrics = {'reply_count': item.get('replyCount', 0), 'repost_count': item.get('repostCount', 0), 'like_count': item.get('likeCount', 0), 'quote_count': item.get('quoteCount', 0)}
        created_at = str(record.get('createdAt', item.get('indexedAt', '')))
        posts.append({'platform': 'bluesky', 'query': query, 'scraped_from': response.url, 'source_url': response.url, 'post_url': post_url, 'x_url': '', 'tweet_id': rkey or uri, 'date_raw': created_at, 'date_iso': created_at, 'username': handle, 'display_name': str(author.get('displayName', '')).strip(), 'original_text': str(record.get('text', '')), 'raw_stats': json.dumps(metrics, sort_keys=True), 'is_retweet': False})
    return (posts, payload.get('cursor'), response.url)

def mastodon_plain_text(content: str) -> str:
    """Convert a Mastodon status's HTML body into readable plain text."""
    return normalize_whitespace(BeautifulSoup(content or '', 'html.parser').get_text(' '))

def date_is_in_range(created_at: str, since: Optional[str], until: Optional[str]) -> bool:
    """Apply optional ISO-like date bounds to APIs without date parameters."""
    if not created_at:
        return True
    try:
        created = dateparser.parse(created_at)
        lower = dateparser.parse(since) if since else None
        upper = dateparser.parse(until) if until else None
        if created is None:
            return True
        if lower and created.date() < lower.date():
            return False
        if upper and created.date() >= upper.date():
            return False
        return True
    except Exception:
        return True

async def fetch_mastodon_page(session: requests.Session, instance_url: str, access_token: str, query: str, max_results: int, since: Optional[str]=None, until: Optional[str]=None, offset: int=0) -> tuple[List[Dict[str, Any]], Optional[int], str]:
    """Fetch one instance-scoped Mastodon full-text status search page."""
    endpoint = f"{instance_url.rstrip('/')}/api/v2/search"
    params: Dict[str, Any] = {'q': query, 'type': 'statuses', 'limit': max(1, min(40, max_results))}
    if access_token and offset:
        params['offset'] = offset
    headers = {'Authorization': f'Bearer {access_token}'} if access_token else {}
    response = await session.get(endpoint, params=params, headers=headers, timeout=60)
    try:
        response.raise_for_status()
        payload = response.json()
    except requests.RequestException as exc:
        raise RuntimeError(f'Mastodon API request failed ({response.status_code}): {response.text[:500]}') from exc
    except ValueError as exc:
        raise RuntimeError('Mastodon returned a non-JSON response.') from exc
    statuses = payload.get('statuses', []) or []
    posts: List[Dict[str, Any]] = []
    for status in statuses:
        is_reblog = bool(status.get('reblog'))
        content_status = status.get('reblog') or status
        account = content_status.get('account') or status.get('account') or {}
        created_at = str(content_status.get('created_at', status.get('created_at', '')))
        if not date_is_in_range(created_at, since, until):
            continue
        metrics = {'reply_count': content_status.get('replies_count', 0), 'reblog_count': content_status.get('reblogs_count', 0), 'favourite_count': content_status.get('favourites_count', 0)}
        post_url = str(content_status.get('url', status.get('url', '')))
        posts.append({'platform': 'mastodon', 'query': query, 'scraped_from': response.url, 'source_url': response.url, 'post_url': post_url, 'x_url': '', 'tweet_id': str(content_status.get('id', status.get('id', ''))), 'date_raw': created_at, 'date_iso': created_at, 'username': str(account.get('acct', account.get('username', ''))), 'display_name': mastodon_plain_text(str(account.get('display_name', ''))), 'original_text': mastodon_plain_text(str(content_status.get('content', ''))), 'raw_stats': json.dumps(metrics, sort_keys=True), 'is_retweet': is_reblog})
    next_offset = None
    if access_token and len(statuses) == params['limit']:
        next_offset = offset + len(statuses)
    return (posts, next_offset, response.url)

def make_x_url_from_nitter_url(nitter_url: str) -> str:
    try:
        parsed = urlparse(nitter_url)
        return 'https://x.com' + parsed.path
    except Exception:
        return ''

def extract_tweet_id_from_url(url: str) -> str:
    match = re.search('/status/(\\d+)', url)
    if match:
        return match.group(1)
    return ''

def get_text_or_empty(element) -> str:
    if element is None:
        return ''
    return element.get_text(' ', strip=True)

async def translate_search_term(client: OpenAI, term: str, target_language: str, model: str, max_retries: int=3) -> str:
    """Translate one query while preserving search syntax where practical."""
    prompt = f'\nTranslate this social-media search term into {target_language}.\n\nReturn only the translated search term, with no quotation marks, label, or explanation.\nPreserve hashtags, @handles, URLs, Boolean operators, and date/filter syntax exactly.\nIf the term is already in {target_language}, return it unchanged.\n\nSearch term:\n{term}\n'.strip()
    for attempt in range(1, max_retries + 1):
        try:
            translated = normalize_whitespace(await generate_llm_text(client=client, model=model, prompt=prompt))
            return translated.strip('"').strip("'")
        except Exception as e:
            if attempt == max_retries:
                print(f'Could not translate {term!r} into {target_language}: {e}. Skipping that query variant.')
                return ''
            await asyncio.sleep(2 * attempt)
    return ''

def debug_nitter_page(html: str, max_chars: int=3000):
    soup = BeautifulSoup(html, 'html.parser')
    title = soup.find('title')
    title_text = title.get_text(' ', strip=True) if title else '[no title]'
    body_text = soup.get_text(' ', strip=True)
    body_text = normalize_whitespace(body_text)
    print('\n--- NITTER PAGE DEBUG ---')
    print('Title:', title_text)
    print('First body text:')
    print(body_text[:max_chars])
    print('\nDetected timeline item count:', len(soup.select('.timeline-item')))
    print('Detected tweet-content count:', len(soup.select('.tweet-content')))
    print('Detected timeline count:', len(soup.select('.timeline')))
    print('Detected show-more count:', len(soup.select('.show-more')))
    print('--- END DEBUG ---\n')

def parse_tweets_from_nitter_html(html: str, page_url: str, query: str) -> List[Dict[str, Any]]:
    soup = BeautifulSoup(html, 'html.parser')
    items = soup.select('.timeline-item')
    parsed = []
    for item in items:
        content_el = item.select_one('.tweet-content')
        if content_el is None:
            continue
        text = content_el.get_text('\n', strip=True)
        text = text.strip()
        if not text:
            continue
        username_el = item.select_one('.username')
        fullname_el = item.select_one('.fullname')
        date_link = item.select_one('.tweet-date a')
        tweet_link = item.select_one('a.tweet-link')
        username = get_text_or_empty(username_el).replace('@', '').strip()
        display_name = get_text_or_empty(fullname_el)
        date_raw = ''
        if date_link is not None:
            date_raw = date_link.get('title') or get_text_or_empty(date_link)
        date_iso = parse_nitter_date(date_raw)
        tweet_href = ''
        if tweet_link is not None:
            tweet_href = tweet_link.get('href') or ''
        nitter_url = urljoin(page_url, tweet_href) if tweet_href else ''
        x_url = make_x_url_from_nitter_url(nitter_url) if nitter_url else ''
        tweet_id = extract_tweet_id_from_url(nitter_url)
        stats_el = item.select_one('.tweet-stats')
        raw_stats = normalize_whitespace(get_text_or_empty(stats_el))
        retweet_header = item.select_one('.retweet-header')
        is_retweet = retweet_header is not None
        parsed.append({'query': query, 'scraped_from': page_url, 'nitter_url': nitter_url, 'x_url': x_url, 'tweet_id': tweet_id, 'date_raw': date_raw, 'date_iso': date_iso, 'username': username, 'display_name': display_name, 'original_text': text, 'raw_stats': raw_stats, 'is_retweet': is_retweet})
    return parsed

async def translate_with_openai(client: OpenAI, text: str, source_lang: str, model: str='gpt-5.6-luna', target_language: str='English', max_retries: int=3) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    if source_lang == 'en' and target_language.lower() == 'english':
        return text
    prompt = f'\nTranslate the following public social media post into {target_language}.\n\nRules:\n- Preserve proper names, institution names, places, hashtags, @handles, emojis, and URLs.\n- Preserve dates and numbers exactly.\n- Do not summarize.\n- Do not add commentary.\n- If the post is already in {target_language}, return it unchanged.\n- If the post contains mixed languages, translate only the non-{target_language} parts.\n- Keep the tone close to the original.\n\nDetected source language: {source_lang}\n\nPost:\n{text}\n'.strip()
    for attempt in range(1, max_retries + 1):
        try:
            return await generate_llm_text(client=client, model=model, prompt=prompt)
        except Exception as e:
            if attempt == max_retries:
                return f'[TRANSLATION ERROR: {e}] {text}'
            wait_time = 2 * attempt
            print(f'LLM translation error. Retrying in {wait_time} seconds...')
            await asyncio.sleep(wait_time)
    return text

def load_json_cache(path: str) -> dict:
    if os.path.exists(path):
        try:
            with open(path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_json_cache(path: str, data: dict):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def get_language_geo_hint(lang_code: str) -> str:
    """
    Give the model a broad geographic prior based on detected language.
    These are intentionally broad and should not be treated as proof.
    """
    lang_code = (lang_code or '').lower().strip()
    hints = {'en': 'English is global and should be treated as a weak location signal only.', 'es': 'Spanish suggests Spain or Latin America; use other evidence to distinguish.', 'fr': 'French suggests France, Belgium, Switzerland, Canada/Quebec, parts of West Africa, North Africa, or other Francophone regions.', 'pt': 'Portuguese suggests Portugal, Brazil, Angola, Mozambique, or other Lusophone regions.', 'it': 'Italian suggests Italy or Italian-language institutional contexts.', 'de': 'German suggests Germany, Austria, Switzerland, or German-language institutional contexts.', 'ru': 'Russian suggests Russia or Russian-speaking post-Soviet regions.', 'ar': 'Arabic suggests Arabic-speaking countries in the Middle East or North Africa.', 'tr': 'Turkish suggests Turkey or Turkish-speaking communities.', 'ja': 'Japanese suggests Japan or Japanese-language institutional contexts.', 'ko': 'Korean suggests South Korea or Korean-language institutional contexts.', 'zh-cn': 'Simplified Chinese suggests Mainland China, Singapore, Malaysia, or simplified-Chinese institutional contexts.', 'zh-tw': 'Traditional Chinese suggests Taiwan, Hong Kong, Macau, or traditional-Chinese institutional contexts.', 'zh': 'Chinese suggests a Chinese-language context; use script, institution names, and other evidence to distinguish Mainland China, Taiwan, Hong Kong, Macau, Singapore, Malaysia, or diaspora contexts.'}
    return hints.get(lang_code, 'No strong language-based geographic prior available.')

async def infer_location_with_openai(client: OpenAI, post: Dict[str, Any], model: str='gpt-5.6-luna', max_retries: int=3) -> Dict[str, Any]:
    """
    Infer a broad public location from tweet/account metadata.

    This intentionally avoids exact street-level locations.
    It should return city/region/country-level places only.
    """
    username = post.get('username', '')
    display_name = post.get('display_name', '')
    query = post.get('query', '')
    original_text = post.get('original_text', '')
    translated_text = post.get('translated_en', '')
    detected_language = post.get('detected_language', 'unknown')
    language_geo_hint = get_language_geo_hint(detected_language)
    prompt = f'\nYou are helping map public institutional social media posts.\n\nInfer the most likely broad public location associated with this post.\nUse only city, region, university, or country-level locations.\nDo NOT infer private addresses, homes, street addresses, or precise personal locations.\n\nReturn ONLY valid JSON with these keys:\n- location_name: string, a geocodable place such as "Madrid, Spain" or "University of Nairobi, Kenya"; use "" if unknown\n- confidence: number between 0 and 1\n- source: string, one of ["language", "username", "display_name", "tweet_text", "query", "combined", "unknown"]\n- reason: short string explaining the evidence\n\nLocation inference rules:\n1. Treat the language of the original post as a major geographic prior.\n2. Use the detected language and script to narrow the likely region before considering weaker clues.\n3. If the original language strongly aligns with a country, region, institution name, or search query, increase confidence.\n4. If the language is widely used across many countries, such as English, French, Spanish, Portuguese, Arabic, or Chinese, do NOT choose a specific country based on language alone.\n5. Prefer explicit place names in the original post text when present.\n6. Next prefer institution names, university names, account names, or handles.\n7. Use the translated text only to understand meaning; prioritize the original post for language, script, and named entities.\n8. If language is the only evidence, return a broad country/region-level location with low confidence, usually 0.15 to 0.35.\n9. If language plus account/institution/text evidence all point to the same place, confidence may be higher.\n10. If no responsible location can be inferred, return location_name "" and confidence 0.\n\nDetected language:\n{detected_language}\n\nLanguage-based geographic prior:\n{language_geo_hint}\n\nMetadata:\nusername: {username}\ndisplay_name: {display_name}\nsearch_query: {query}\n\nOriginal post:\n{original_text}\n\nTranslated post:\n{translated_text}\n'.strip()
    for attempt in range(1, max_retries + 1):
        try:
            text = await generate_llm_text(client=client, model=model, prompt=prompt)
            text = text.replace('```json', '').replace('```', '').strip()
            data = json.loads(text)
            return {'location_name': str(data.get('location_name', '')).strip(), 'confidence': float(data.get('confidence', 0) or 0), 'source': str(data.get('source', 'unknown')).strip(), 'reason': str(data.get('reason', '')).strip()}
        except Exception as e:
            if attempt == max_retries:
                return {'location_name': '', 'confidence': 0.0, 'source': 'unknown', 'reason': f'Location inference failed: {e}'}
            await asyncio.sleep(2 * attempt)
    return {'location_name': '', 'confidence': 0.0, 'source': 'unknown', 'reason': 'Location inference failed.'}

async def enrich_records_with_locations(records: List[PostRecord], openai_model: str='gpt-5.6-luna') -> List[PostRecord]:
    """
    Infer broad locations and geocode them.
    """
    if not records:
        return records
    if not INFER_LOCATIONS:
        return records
    print('\nInferring and geocoding locations...')
    client = create_llm_client()
    geolocator = None
    geocode_cache = load_json_cache(GEOCODE_CACHE_FILE)
    enriched_records = []
    iterator = records
    if tqdm is not None:
        iterator = tqdm(records)
    for record in iterator:
        post_dict = asdict(record)
        inference = await infer_location_with_openai(client=client, post=post_dict, model=openai_model)
        inferred_location = inference.get('location_name', '')
        confidence = float(inference.get('confidence', 0) or 0)
        source = inference.get('source', 'unknown')
        reason = inference.get('reason', '')
        latitude = ''
        longitude = ''
        geocode_display_name = ''
        if inferred_location and confidence >= MIN_LOCATION_CONFIDENCE:
            geocode = await geocode_location(location_name=inferred_location, cache=geocode_cache, geolocator=geolocator, delay_seconds=GEOCODE_DELAY_SECONDS)
            latitude = geocode.get('latitude', '')
            longitude = geocode.get('longitude', '')
            geocode_display_name = geocode.get('geocode_display_name', '')
        record.inferred_location = inferred_location
        record.location_confidence = confidence
        record.location_source = source
        record.location_reason = reason
        record.latitude = latitude
        record.longitude = longitude
        record.geocode_display_name = geocode_display_name
        enriched_records.append(record)
    return enriched_records

def create_tweet_map(df: pd.DataFrame, map_file: str=f'social_search_map_{RUN_TIMESTAMP}.html'):
    """
    Create an interactive Folium map with popups showing original and translated tweets.

    Tile fix:
    - Do not use Folium's direct OpenStreetMap default tile layer.
    - That default can trigger 503r / Access Blocked errors from the OSM tile server.
    - Use CARTO basemap tiles instead, with proper attribution.
    """
    if df is None or df.empty:
        print('No dataframe available for mapping.')
        return None
    if 'latitude' not in df.columns or 'longitude' not in df.columns:
        print('No latitude/longitude columns found. Skipping map.')
        return None
    map_df = df.copy()

    def map_coordinate_numbers(series: pd.Series) -> pd.Series:
        normalized = series.astype('string').str.strip().str.replace("^'(?=[+-]?\\d)", '', regex=True)
        return pd.to_numeric(normalized, errors='coerce')
    map_df['latitude_num'] = map_coordinate_numbers(map_df['latitude'])
    map_df['longitude_num'] = map_coordinate_numbers(map_df['longitude'])
    map_df['activity_date'] = pd.to_datetime(map_df.get('date_iso'), errors='coerce', utc=True)
    map_df = map_df.dropna(subset=['latitude_num', 'longitude_num'])
    if map_df.empty:
        print('No geocoded rows available for mapping.')
        return None
    center_lat = map_df['latitude_num'].mean()
    center_lon = map_df['longitude_num'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=2, tiles=None, control_scale=True)
    folium.TileLayer(tiles=MAP_BASE_TILE_URL, attr=MAP_BASE_TILE_ATTRIBUTION, name=MAP_BASE_TILE_NAME, overlay=False, control=True).add_to(m)
    if ADD_ALTERNATE_BASEMAPS:
        folium.TileLayer(tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png', attr=MAP_BASE_TILE_ATTRIBUTION, name='CartoDB Dark Matter', overlay=False, control=True).add_to(m)
        folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', attr='Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community', name='Esri World Imagery', overlay=False, control=True).add_to(m)
    cluster = MarkerCluster(name='Inferred tweet locations').add_to(m)
    reference_time = pd.Timestamp.now(tz='UTC')
    heatmap_windows = (7, 30, 90, 365)
    heatmap_counts = {}
    for days in heatmap_windows:
        cutoff = reference_time - pd.Timedelta(days=days)
        recent = map_df[map_df['activity_date'].notna() & (map_df['activity_date'] >= cutoff) & (map_df['activity_date'] <= reference_time)].copy()
        heatmap_counts[days] = len(recent)
        recent['heat_lat'] = recent['latitude_num'].round(2)
        recent['heat_lon'] = recent['longitude_num'].round(2)
        weighted_locations = recent.groupby(['heat_lat', 'heat_lon'], as_index=False).size().rename(columns={'size': 'weight'})
        heat_data = weighted_locations[['heat_lat', 'heat_lon', 'weight']].values.tolist()
        heat_layer = folium.FeatureGroup(name=f'Activity heatmap — last {days} days ({len(recent)} posts)', overlay=True, control=True, show=days == 30)
        if heat_data:
            HeatMap(heat_data, min_opacity=0.25, radius=24, blur=18, max_zoom=8, gradient={0.2: '#2c7bb6', 0.4: '#00a6ca', 0.6: '#ffff8c', 0.8: '#fdae61', 1.0: '#d7191c'}).add_to(heat_layer)
        heat_layer.add_to(m)
    activity_summary = ' · '.join((f'{days}d: {heatmap_counts[days]}' for days in heatmap_windows))
    folium.Element(f"""<div style="position: fixed; bottom: 28px; left: 28px; z-index: 9999;\n        background: white; color: #222; border: 1px solid #777; border-radius: 4px;\n        padding: 9px 11px; width: 245px; font: 12px/1.35 Arial, sans-serif;\n        box-shadow: 0 1px 4px #777;">\n        <strong>Geocoded activity heatmaps</strong><br>{activity_summary}<br>\n        <div style="margin-top: 7px;">Relative activity intensity</div>\n        <div style="height: 10px; margin-top: 3px; border: 1px solid #888;\n        background: linear-gradient(to right, #2c7bb6 0%, #00a6ca 25%,\n        #ffff8c 50%, #fdae61 75%, #d7191c 100%);"></div>\n        <div style="display: flex; justify-content: space-between; color: #444;">\n        <span>Lower</span><span>Higher</span>\n        </div>\n        <div style="margin: 5px 0; color: #555; font-size: 11px;">\n        Colors show relative concentration within each selected time layer;\n        they are not fixed post-count ranges. Use the layer control to switch\n        between 7, 30, 90, and 365 days.\n        </div>\n        <span style="color:#555">As of {reference_time.strftime('%Y-%m-%d %H:%M UTC')}</span>\n        </div>""").add_to(m.get_root().html)
    for _, row in map_df.iterrows():
        original_text = html_lib.escape(str(row.get('original_text', '')))
        translated_en = html_lib.escape(str(row.get('translated_en', '')))
        username = html_lib.escape(str(row.get('username', '')))
        display_name = html_lib.escape(str(row.get('display_name', '')))
        date_raw = html_lib.escape(str(row.get('date_raw', '')))
        inferred_location = html_lib.escape(str(row.get('inferred_location', '')))
        confidence = html_lib.escape(str(row.get('location_confidence', '')))
        source = html_lib.escape(str(row.get('location_source', '')))
        reason = html_lib.escape(str(row.get('location_reason', '')))
        post_url = html_lib.escape(safe_http_url(row.get('post_url', row.get('x_url', ''))))
        source_mode = str(row.get('source_mode', ''))
        source_url = html_lib.escape(safe_http_url(row.get('source_url', '')))
        legacy_link = f'<div><a href="{source_url}" target="_blank">Open saved Nitter source</a></div>' if source_mode == 'saved_html' and source_url else ''
        popup_html = f'\n        <div style="width: 380px; font-family: Arial, sans-serif;">\n            <h4 style="margin-bottom: 4px;">@{username}</h4>\n            <div><strong>Name:</strong> {display_name}</div>\n            <div><strong>Date:</strong> {date_raw}</div>\n            <div><strong>Inferred location:</strong> {inferred_location}</div>\n            <div><strong>Confidence:</strong> {confidence}</div>\n            <div><strong>Source:</strong> {source}</div>\n            <div><strong>Reason:</strong> {reason}</div>\n            <hr>\n            <div><strong>Original:</strong></div>\n            <div style="white-space: pre-wrap; margin-bottom: 8px;">{original_text}</div>\n            <div><strong>Translation:</strong></div>\n            <div style="white-space: pre-wrap; margin-bottom: 8px;">{translated_en}</div>\n            <hr>\n            <div><a href="{post_url}" target="_blank">Open original post</a></div>\n            {legacy_link}\n        </div>\n        '
        tooltip = f'@{username} — {inferred_location}'
        folium.Marker(location=[row['latitude_num'], row['longitude_num']], popup=folium.Popup(popup_html, max_width=440), tooltip=tooltip).add_to(cluster)
    folium.LayerControl().add_to(m)
    m.save(map_file)
    print(f'Map saved to: {map_file}')
    print('Map tile layer: CARTO basemap CDN, not the direct OpenStreetMap tile server.')
    return m

async def make_record_from_post(post: Dict[str, Any], source_mode: str, source_host: str, client: Optional[OpenAI], openai_model: str, target_language: str, translate: bool) -> PostRecord:
    original_text = post.get('original_text', '')
    detected_language = detect_language_safe(original_text)
    if translate and client is not None:
        translated_en = await translate_with_openai(client=client, text=original_text, source_lang=detected_language, model=openai_model, target_language=target_language)
        await asyncio.sleep(TRANSLATION_DELAY_SECONDS)
    else:
        translated_en = ''
    return PostRecord(platform=post.get('platform', 'legacy'), query=post.get('query', ''), source_mode=source_mode, source_host=source_host, source_url=post.get('source_url', post.get('nitter_url', '')), scraped_from=post.get('scraped_from', ''), post_url=post.get('post_url', post.get('x_url', '')), x_url=post.get('x_url', ''), tweet_id=post.get('tweet_id', ''), date_raw=post.get('date_raw', ''), date_iso=post.get('date_iso', ''), username=post.get('username', ''), display_name=post.get('display_name', ''), detected_language=detected_language, original_text=original_text, translated_en=translated_en, raw_stats=post.get('raw_stats', ''), is_retweet=post.get('is_retweet', False), inferred_location='', location_confidence=0.0, location_source='', location_reason='', latitude='', longitude='', geocode_display_name='')

async def run_x_api_collection(bearer_token: str, search_mode: str='recent', since: Optional[str]=None, until: Optional[str]=None, max_posts_per_query: int=5, max_pages_per_query: int=1, output_file: str=f'social_search_posts_{RUN_TIMESTAMP}.csv', handles: Optional[List[str]]=None, search_terms: Optional[List[str]]=None, post_languages: Optional[List[str]]=None, include_retweets: bool=False, target_language: str='English', openai_model: str='gpt-5.6-luna', translate: bool=False):
    print(f'Official X API {search_mode} collection started.')
    session = create_requests_session()
    if translate:
        print('Translation is ON.')
        client = create_llm_client()
    else:
        print('Translation is OFF.')
        client = None
    if search_terms is None:
        search_terms = DEFAULT_SEARCH_TERMS
    queries = build_queries(terms=search_terms, handles=handles)
    print(f'\nBuilt {len(queries)} queries:')
    for q in queries:
        print('  ', q)
    records: List[PostRecord] = []
    seen_ids = set()
    seen_text_fallbacks = set()
    for query in queries:
        print('\n' + '=' * 70)
        api_query = build_x_query(query, include_retweets=include_retweets, languages=post_languages)
        print(f'Searching X for: {api_query}')
        posts_collected_for_query = 0
        next_token = None
        for page_num in range(1, max_pages_per_query + 1):
            print(f'Page {page_num}/{max_pages_per_query}')
            remaining = max_posts_per_query - posts_collected_for_query
            try:
                parsed_posts, next_token, request_url = await fetch_x_api_page(session=session, bearer_token=bearer_token, query=api_query, max_results=remaining, search_mode=search_mode, since=since, until=until, next_token=next_token)
            except RuntimeError as exc:
                print(f'X API error: {exc}')
                break
            print(f'X returned {len(parsed_posts)} posts.')
            for post in parsed_posts:
                tweet_id = post.get('tweet_id', '')
                fallback_key = (post.get('username', ''), post.get('date_raw', ''), post.get('original_text', '')[:120])
                if tweet_id and tweet_id in seen_ids:
                    continue
                if not tweet_id and fallback_key in seen_text_fallbacks:
                    continue
                if tweet_id:
                    seen_ids.add(tweet_id)
                else:
                    seen_text_fallbacks.add(fallback_key)
                record = await make_record_from_post(post=post, source_mode=f'x_api_{search_mode}', source_host='api.x.com', client=client, openai_model=openai_model, target_language=target_language, translate=translate)
                records.append(record)
                posts_collected_for_query += 1
                print(f'Collected {posts_collected_for_query}/{max_posts_per_query} for this query: @{record.username} {record.date_raw}')
                if posts_collected_for_query >= max_posts_per_query:
                    break
            if posts_collected_for_query >= max_posts_per_query:
                print('Reached max posts for this query.')
                break
            if not next_token:
                print('No additional result page is available.')
                break
            print(f'Sleeping {REQUEST_DELAY_SECONDS} seconds before next page...')
            await asyncio.sleep(REQUEST_DELAY_SECONDS)
    return records

async def run_bluesky_collection(since: Optional[str]=None, until: Optional[str]=None, max_posts_per_query: int=5, max_pages_per_query: int=1, search_terms: Optional[List[str]]=None, target_language: str='English', openai_model: str='gpt-5.6-luna', translate: bool=False, access_jwt: str='') -> List[PostRecord]:
    """Collect public Bluesky posts and normalize them into PostRecord objects."""
    mode_label = 'authenticated PDS proxy' if access_jwt else 'public API'
    print(f'Bluesky {mode_label} collection started.')
    session = create_requests_session()
    client = create_llm_client() if translate else None
    queries = build_queries(terms=search_terms or DEFAULT_SEARCH_TERMS)
    records: List[PostRecord] = []
    seen_ids = set()
    for query in queries:
        print('\n' + '=' * 70)
        print(f'Searching Bluesky for: {query}')
        collected = 0
        cursor = None
        for page_num in range(1, max_pages_per_query + 1):
            try:
                posts, cursor, _ = await fetch_bluesky_page(session=session, query=query, max_results=max_posts_per_query - collected, since=since, until=until, cursor=cursor, access_jwt=access_jwt)
            except RuntimeError as exc:
                print(f'Bluesky API error: {exc}')
                break
            print(f'Bluesky returned {len(posts)} posts on page {page_num}.')
            for post in posts:
                post_id = post.get('tweet_id', '')
                if post_id and ('bluesky', post_id) in seen_ids:
                    continue
                if post_id:
                    seen_ids.add(('bluesky', post_id))
                record = await make_record_from_post(post=post, source_mode='api', source_host='bsky.social' if access_jwt else 'public.api.bsky.app', client=client, openai_model=openai_model, target_language=target_language, translate=translate)
                records.append(record)
                collected += 1
                print(f'Collected {collected}/{max_posts_per_query} from Bluesky: @{record.username} {record.date_raw}')
                if collected >= max_posts_per_query:
                    break
            if collected >= max_posts_per_query or not cursor:
                break
            await asyncio.sleep(REQUEST_DELAY_SECONDS)
    return records

async def run_mastodon_collection(instance_url: str, access_token: str, since: Optional[str]=None, until: Optional[str]=None, max_posts_per_query: int=5, max_pages_per_query: int=1, search_terms: Optional[List[str]]=None, include_reblogs: bool=False, target_language: str='English', openai_model: str='gpt-5.6-luna', translate: bool=False) -> List[PostRecord]:
    """Collect searchable public statuses known to one Mastodon instance."""
    print(f'Mastodon API collection started for {instance_url}.')
    session = create_requests_session()
    client = create_llm_client() if translate else None
    queries = build_queries(terms=search_terms or DEFAULT_SEARCH_TERMS)
    records: List[PostRecord] = []
    seen_ids = set()
    for query in queries:
        print('\n' + '=' * 70)
        print(f'Searching Mastodon on {instance_url} for: {query}')
        collected = 0
        offset = 0
        for page_num in range(1, max_pages_per_query + 1):
            try:
                posts, next_offset, _ = await fetch_mastodon_page(session=session, instance_url=instance_url, access_token=access_token, query=query, max_results=max_posts_per_query - collected, since=since, until=until, offset=offset)
            except RuntimeError as exc:
                print(f'Mastodon API error: {exc}')
                break
            print(f'Mastodon returned {len(posts)} usable posts on page {page_num}.')
            for post in posts:
                if not include_reblogs and post.get('is_retweet'):
                    continue
                post_id = post.get('tweet_id', '')
                dedupe_key = (instance_url, post_id)
                if post_id and dedupe_key in seen_ids:
                    continue
                if post_id:
                    seen_ids.add(dedupe_key)
                record = await make_record_from_post(post=post, source_mode='api', source_host=urlparse(instance_url).netloc, client=client, openai_model=openai_model, target_language=target_language, translate=translate)
                records.append(record)
                collected += 1
                print(f'Collected {collected}/{max_posts_per_query} from Mastodon: @{record.username} {record.date_raw}')
                if collected >= max_posts_per_query:
                    break
            if collected >= max_posts_per_query or next_offset is None:
                break
            offset = next_offset
            await asyncio.sleep(REQUEST_DELAY_SECONDS)
    return records

async def run_saved_html_collection(html_files: List[str], output_file: str, include_retweets: bool=False, target_language: str='English', openai_model: str='gpt-5.6-luna', translate: bool=False):
    print('Saved HTML collection started.')
    if translate:
        print('Translation is ON.')
        client = create_llm_client()
    else:
        print('Translation is OFF.')
        client = None
    records: List[PostRecord] = []
    seen_ids = set()
    seen_text_fallbacks = set()
    for html_file in html_files:
        print('\n' + '=' * 70)
        print(f'Parsing saved HTML file: {html_file}')
        if not os.path.exists(html_file):
            print(f'File not found: {html_file}')
            continue
        with open(html_file, 'r', encoding='utf-8', errors='replace') as f:
            html = f.read()
        debug_nitter_page(html, max_chars=1500)
        parsed_posts = parse_tweets_from_nitter_html(html=html, page_url='https://nitter.net/search', query=f'saved_html:{html_file}')
        print(f'Parsed {len(parsed_posts)} posts from saved HTML.')
        for post in parsed_posts:
            if not include_retweets and post.get('is_retweet'):
                print('Skipping retweet.')
                continue
            tweet_id = post.get('tweet_id', '')
            fallback_key = (post.get('username', ''), post.get('date_raw', ''), post.get('original_text', '')[:120])
            if tweet_id and tweet_id in seen_ids:
                continue
            if not tweet_id and fallback_key in seen_text_fallbacks:
                continue
            if tweet_id:
                seen_ids.add(tweet_id)
            else:
                seen_text_fallbacks.add(fallback_key)
            record = await make_record_from_post(post=post, source_mode='saved_html', source_host='saved_html', client=client, openai_model=openai_model, target_language=target_language, translate=translate)
            records.append(record)
            print(f'Collected saved HTML post: @{record.username} {record.date_raw}')
    return records

async def save_records(records: List[PostRecord], output_file: str):
    if not records:
        print('\nNo records collected.')
        print('Likely causes:')
        print('1. The selected APIs returned no matching accessible posts.')
        print('2. The query is too narrow.')
        print('3. Date filters exclude available results.')
        print('4. A credential, API access, billing, or rate-limit error occurred.')
        print('5. Mastodon full-text search is unavailable on the chosen server.')
        print('6. In legacy mode, saved HTML contains no visible timeline items.')
        return None
    if CREATE_MAP or INFER_LOCATIONS:
        records = await enrich_records_with_locations(records=records, openai_model=LLM_MODEL)
    df = pd.DataFrame([asdict(r) for r in records])
    numeric_columns = {'latitude', 'longitude', 'location_confidence'}
    for col in df.columns:
        protect_formulas = col not in numeric_columns
        df[col] = df[col].apply(lambda value: clean_cell(value, prevent_formula_injection=protect_formulas))
    preferred_order = ['platform', 'date_iso', 'date_raw', 'username', 'display_name', 'detected_language', 'inferred_location', 'location_confidence', 'location_source', 'location_reason', 'latitude', 'longitude', 'geocode_display_name', 'original_text', 'translated_en', 'post_url', 'x_url', 'tweet_id', 'raw_stats', 'is_retweet', 'query', 'source_mode', 'source_host', 'source_url', 'scraped_from']
    existing_order = [col for col in preferred_order if col in df.columns]
    remaining_cols = [col for col in df.columns if col not in existing_order]
    df = df[existing_order + remaining_cols]
    for col in numeric_columns & set(df.columns):
        df[col] = pd.to_numeric(df[col], errors='coerce')
    if output_file.lower().endswith('.csv'):
        csv_file = output_file
        xlsx_file = output_file[:-4] + '.xlsx'
    else:
        csv_file = output_file + '.csv'
        xlsx_file = output_file + '.xlsx'
    df.to_csv(csv_file, index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL, lineterminator='\n')
    with pd.ExcelWriter(xlsx_file, engine='openpyxl') as writer:
        df.to_excel(writer, index=False, sheet_name='posts')
        worksheet = writer.sheets['posts']
        worksheet.freeze_panes = 'A2'
        worksheet.auto_filter.ref = worksheet.dimensions
        header_fill = PatternFill('solid', fgColor='D9EAF7')
        header_font = Font(bold=True)
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        width_by_column_name = {'platform': 14, 'date_iso': 24, 'date_raw': 28, 'username': 20, 'display_name': 30, 'detected_language': 18, 'inferred_location': 34, 'location_confidence': 18, 'location_source': 18, 'location_reason': 60, 'latitude': 18, 'longitude': 18, 'geocode_display_name': 60, 'original_text': 90, 'translated_en': 90, 'post_url': 60, 'x_url': 60, 'tweet_id': 24, 'raw_stats': 30, 'is_retweet': 14, 'query': 30, 'source_mode': 16, 'source_host': 24, 'source_url': 60, 'scraped_from': 60}
        for idx, column_name in enumerate(df.columns, start=1):
            col_letter = get_column_letter(idx)
            worksheet.column_dimensions[col_letter].width = width_by_column_name.get(column_name, 24)
        for row in worksheet.iter_rows():
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
        for row_idx in range(2, worksheet.max_row + 1):
            worksheet.row_dimensions[row_idx].height = 60
    print(f'\nDone. Wrote {len(df)} records.')
    print(f'CSV file:   {csv_file}')
    print(f'Excel file: {xlsx_file}')
    if CREATE_MAP:
        create_tweet_map(df=df, map_file=MAP_FILE)
    print('\nPreview:')
    print(df.head())
    return df

def load_results_file(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    if path.suffix.casefold() == '.xlsx':
        return pd.read_excel(path, sheet_name='posts')
    return pd.read_csv(path)

In [ ]:
import ipywidgets as widgets

# Re-running this UI cell must not start a second job beside an existing one.
if '_sugar_task' in globals() and _sugar_task is not None and not _sugar_task.done():
    raise RuntimeError('Cancel the running SUGAR job before re-running this cell.')
if '_sugar_ui' in globals():
    globals()['_sugar_ui'].close()

_sugar_task = None
_sugar_df = None
_sugar_files = []

STYLE = {'description_width':'initial'}
FULL = widgets.Layout(width='100%')
terms = widgets.Textarea(value='Democracy', description='Search terms (one per line)', layout=FULL, style=STYLE)
sources = widgets.SelectMultiple(options=[('Bluesky','bluesky'),('Mastodon','mastodon'),('X','x')], value=('bluesky',), description='Sources', style=STYLE)
query_languages = widgets.Text(description='Translate queries into', placeholder='Optional: Spanish, French', layout=FULL, style=STYLE)
x_languages = widgets.Text(description='X post language codes', placeholder='Optional: en, es, fr', layout=FULL, style=STYLE)
handles = widgets.Text(description='X handles (optional)', placeholder='Comma-separated; overrides X keyword queries', layout=FULL, style=STYLE)
since = widgets.Text(description='On or after', placeholder='YYYY-MM-DD', style=STYLE)
until = widgets.Text(description='Before (exclusive)', placeholder='YYYY-MM-DD', style=STYLE)
post_limit = widgets.BoundedIntText(value=10,min=1,max=500,description='Posts/query/source',style=STYLE)
page_limit = widgets.BoundedIntText(value=1,min=1,max=20,description='Pages/query/source',style=STYLE)
retweets = widgets.Checkbox(value=False,description='Include reposts / retweets',style=STYLE)
x_mode = widgets.Dropdown(options=[('Recent','recent'),('Full archive (requires API access)','all')],description='X search',style=STYLE)
x_key = widgets.Password(description='X bearer token',layout=FULL,style=STYLE)
bsky_handle = widgets.Text(description='Bluesky handle',layout=FULL,style=STYLE)
bsky_password = widgets.Password(description='Bluesky app password',layout=FULL,style=STYLE)
masto_instance = widgets.Text(value='https://mastodon.social',description='Mastodon instance',layout=FULL,style=STYLE)
masto_key = widgets.Password(description='Mastodon access token',layout=FULL,style=STYLE)
provider = widgets.Dropdown(options=[('Virginia Tech ARC','arc'),('OpenAI','openai')],value='arc',description='LLM provider',style=STYLE)
model = widgets.Text(value='gpt-oss-120b',description='Model ID',layout=FULL,style=STYLE)
llm_key = widgets.Password(description='LLM API key',layout=FULL,style=STYLE)
translate = widgets.Checkbox(value=False,description='Translate collected / loaded posts',style=STYLE)
target = widgets.Text(value='English',description='Translation language',layout=FULL,style=STYLE)
infer = widgets.Checkbox(value=False,description='Infer broad public locations (LLM)',style=STYLE)
geocode = widgets.Checkbox(value=False,description='Geocode inferred locations (Nominatim)',style=STYLE)
geocode_endpoint = widgets.Text(value=GEOCODE_ENDPOINT,description='Geocoder search URL',layout=FULL,style=STYLE)
confidence = widgets.FloatSlider(value=.2,min=0,max=1,step=.05,description='Minimum location confidence',style=STYLE,layout=FULL)
upload = widgets.FileUpload(accept='.csv,.xlsx,.html,.htm',multiple=True,description='Upload data')
file_path = widgets.Text(description='Or workspace file',placeholder='Optional: existing-file.csv',style=STYLE,layout=FULL)
report_format = widgets.Dropdown(options=[('HTML + Word + print-to-PDF','both'),('HTML + print-to-PDF','html'),('Word (.docx)','docx')],value='both',description='Report format',style=STYLE)
status = widgets.HTML('<b>Ready.</b> Start with Load demo data or upload a results file.')
log = widgets.Output(layout=widgets.Layout(border='1px solid #ddd',max_height='280px',overflow='auto'))
results_view = widgets.Output()
downloads_view = widgets.Output()
search_button = widgets.Button(description='Search selected sources',button_style='primary',icon='search')
load_button = widgets.Button(description='Load uploaded / workspace file',icon='upload',layout=widgets.Layout(width='260px'))
demo_button = widgets.Button(description='Load demo data',icon='flask')
enrich_button = widgets.Button(description='Translate / enrich loaded data',layout=widgets.Layout(width='250px'))
export_button = widgets.Button(description='Export CSV + Excel',icon='download')
map_button = widgets.Button(description='Create map',icon='map')
report_button = widgets.Button(description='Create report',icon='file-text')
cancel_button = widgets.Button(description='Cancel',button_style='warning',disabled=True)
action_buttons = [search_button,load_button,demo_button,enrich_button,export_button,map_button,report_button]
secret_fields = [llm_key,x_key,bsky_password,masto_key]
input_controls = [terms,sources,query_languages,x_languages,handles,since,until,post_limit,page_limit,retweets,x_mode,x_key,bsky_handle,bsky_password,masto_instance,masto_key,provider,model,llm_key,translate,target,infer,geocode,geocode_endpoint,confidence,upload,file_path,report_format]


def set_status(text):
    status.value = '<b>' + html_lib.escape(text) + '</b>'


def normalise_frame(df):
    df = df.copy().fillna('')
    if 'original_text' not in df.columns:
        for alternative in ('text','content','post'):
            if alternative in df.columns:
                df['original_text'] = df[alternative]
                break
    for field in fields(PostRecord):
        if field.name not in df.columns:
            df[field.name] = False if field.name=='is_retweet' else (0.0 if field.name=='location_confidence' else '')
    for col in ['original_text','translated_en','username','display_name','platform','query','date_iso']:
        df[col] = df[col].astype(str)
    df['location_confidence'] = pd.to_numeric(df['location_confidence'], errors='coerce').fillna(0.0)
    return df


def records_from_frame(df):
    names = [f.name for f in fields(PostRecord)]
    records = []
    for row in normalise_frame(df)[names].to_dict('records'):
        row['is_retweet'] = str(row['is_retweet']).lower() in ('true','1','yes')
        try: row['location_confidence'] = float(row['location_confidence'] or 0)
        except (ValueError,TypeError): row['location_confidence'] = 0.0
        records.append(PostRecord(**row))
    return records


def preview():
    with results_view:
        results_view.clear_output(wait=True)
        if _sugar_df is not None:
            print(f'{len(_sugar_df)} rows loaded. Showing at most 20 below.')
            display(_sugar_df.head(20))


def offer_files(paths):
    """Use self-contained download links; no assumption about a /files HTTP server."""
    global _sugar_files
    _sugar_files = [str(p) for p in paths if Path(p).is_file()]
    with downloads_view:
        downloads_view.clear_output(wait=True)
        for filename in _sugar_files:
            p=Path(filename)
            mime={'.csv':'text/csv','.xlsx':'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet',
                  '.docx':'application/vnd.openxmlformats-officedocument.wordprocessingml.document',
                  '.html':'text/html'}.get(p.suffix,'application/octet-stream')
            encoded=base64.b64encode(p.read_bytes()).decode('ascii')
            display(HTML(f'<p><a download="{html_lib.escape(p.name,quote=True)}" href="data:{mime};base64,{encoded}">Download {html_lib.escape(p.name)}</a></p>'))
        if _sugar_files:
            print('Files are also in sugar-output in JupyterLite. Download them to keep a copy on your computer.')


def timestamp():
    return datetime.now().strftime('%Y%m%d_%H%M%S_%f')


def validate_settings(needs_llm=False):
    global LLM_PROVIDER, LLM_MODEL, LLM_API_KEY, LLM_BASE_URL, INFER_LOCATIONS, GEOCODE_LOCATIONS, MIN_LOCATION_CONFIDENCE, GEOCODE_ENDPOINT
    LLM_PROVIDER=provider.value
    LLM_MODEL=model.value.strip()
    LLM_API_KEY=llm_key.value.strip()
    LLM_BASE_URL=ARC_BASE_URL if LLM_PROVIDER=='arc' else 'https://api.openai.com/v1'
    INFER_LOCATIONS=infer.value
    GEOCODE_LOCATIONS=geocode.value
    GEOCODE_ENDPOINT=geocode_endpoint.value.strip()
    if GEOCODE_LOCATIONS and urlparse(GEOCODE_ENDPOINT).scheme != "https": raise ValueError("Use an HTTPS geocoding endpoint.")
    MIN_LOCATION_CONFIDENCE=confidence.value
    if needs_llm and (not LLM_API_KEY or not LLM_MODEL):
        raise ValueError('Enter your LLM API key and a valid model ID for translation or location inference.')
    if not target.value.strip():
        raise ValueError('Enter a target language.')
    for control in (since,until):
        if control.value.strip(): datetime.strptime(control.value.strip(),'%Y-%m-%d')
    if since.value.strip() and until.value.strip() and since.value.strip()>=until.value.strip():
        raise ValueError('The end date must be later than the start date.')


async def do_search():
    global _sugar_df
    languages=[v.strip() for v in query_languages.value.split(',') if v.strip()]
    validate_settings(translate.value or infer.value or bool(languages))
    queries=[v.strip() for v in terms.value.splitlines() if v.strip()]
    if not queries: raise ValueError('Enter at least one search term.')
    if not sources.value: raise ValueError('Select at least one source.')
    if 'x' in sources.value and not x_key.value.strip(): raise ValueError('Enter an X bearer token or deselect X.')
    instance=masto_instance.value.strip().rstrip('/')
    if 'mastodon' in sources.value and (urlparse(instance).scheme!='https' or not urlparse(instance).netloc):
        raise ValueError('Mastodon requires an HTTPS instance URL.')
    if bool(bsky_handle.value.strip()) != bool(bsky_password.value):
        raise ValueError('Provide both a Bluesky handle and an app password, or leave both blank.')
    _sugar_df=None  # Never present previous results as a failed search's results.
    results_view.clear_output();downloads_view.clear_output()
    if languages:
        client=create_llm_client()
        for term in list(queries):
            for language in languages:
                translated=await translate_search_term(client,term,language,LLM_MODEL)
                if translated and translated.casefold() not in {q.casefold() for q in queries}: queries.append(translated)
    print('Queries:',queries)
    shared=dict(since=since.value.strip() or None,until=until.value.strip() or None,
                max_posts_per_query=post_limit.value,max_pages_per_query=page_limit.value,
                search_terms=queries,target_language=target.value.strip(),openai_model=LLM_MODEL,translate=translate.value)
    records=[]
    if 'x' in sources.value:
        codes=[v.strip() for v in x_languages.value.split(',') if v.strip()]
        if any(not re.fullmatch('[A-Za-z-]{2,15}',v) for v in codes): raise ValueError('Invalid X language code.')
        records.extend(await run_x_api_collection(bearer_token=x_key.value.strip(),search_mode=x_mode.value,
            output_file='',handles=[h.strip() for h in handles.value.split(',') if h.strip()] or None,
            post_languages=codes,include_retweets=retweets.value,**shared))
    if 'bluesky' in sources.value:
        token=''
        if bsky_handle.value.strip():
            token=await create_bluesky_access_token(BrowserSession(),bsky_handle.value.strip(),bsky_password.value)
        records.extend(await run_bluesky_collection(access_jwt=token,**shared))
        token=''
    if 'mastodon' in sources.value:
        records.extend(await run_mastodon_collection(instance_url=instance,access_token=masto_key.value.strip(),
            include_reblogs=retweets.value,**shared))
    if not records:
        raise ValueError('No posts collected. Read the source messages above for API/CORS errors or empty results.')
    path=OUTPUT_DIR/f'social_search_posts_{timestamp()}.csv'
    _sugar_df=await save_records(records,str(path))
    preview();offer_files([path,path.with_suffix('.xlsx')])


def uploaded_paths():
    value=upload.value
    entries=list(value.values()) if isinstance(value,dict) else list(value)
    paths=[]
    folder=Path('sugar-uploads');folder.mkdir(exist_ok=True)
    for item in entries:
        raw_name=item.get('name') or item.get('metadata',{}).get('name','upload.csv')
        name=Path(str(raw_name).replace('\\','/')).name
        path=folder/name
        if path.suffix.lower() not in {'.csv','.xlsx','.html','.htm'}: raise ValueError('Upload CSV, XLSX, or saved HTML files.')
        path.write_bytes(bytes(item['content']));paths.append(path)
    if file_path.value.strip():
        path=Path(file_path.value.strip())
        if not path.is_file(): raise ValueError('Workspace file not found. Upload your computer’s file first.')
        paths.append(path)
    return paths


async def do_load():
    global _sugar_df
    paths=uploaded_paths()
    if not paths: raise ValueError('Upload a file or enter its JupyterLite workspace path.')
    frames=[]
    for p in paths:
        if p.suffix.lower() in {'.html','.htm'}:
            # HTML is parsed as data, never displayed or executed.
            posts=parse_tweets_from_nitter_html(p.read_text(encoding='utf-8',errors='replace'),'https://nitter.net',f'saved_html:{p.name}')
            if not posts: print(f'No saved Nitter posts found in {p.name}.');continue
            records=[await make_record_from_post(post,'saved_html','saved_html',None,LLM_MODEL,target.value,False) for post in posts]
            frames.append(pd.DataFrame([asdict(r) for r in records]))
        elif p.suffix.lower()=='.csv': frames.append(pd.read_csv(p,dtype=str,keep_default_na=False))
        elif p.suffix.lower()=='.xlsx': frames.append(pd.read_excel(p,dtype=str,keep_default_na=False))
        else: raise ValueError('Supported files: CSV, XLSX, saved Nitter HTML.')
    if not frames: raise ValueError('No readable records found.')
    _sugar_df=normalise_frame(pd.concat(frames,ignore_index=True))
    print(f'Loaded {len(_sugar_df)} rows. Loading does not make API requests.')
    preview()


async def do_demo():
    global _sugar_df
    _sugar_df=normalise_frame(pd.DataFrame([
        {'platform':'demo','username':'example_es','date_iso':datetime.now().isoformat()+'Z',
         'original_text':'La inteligencia artificial puede apoyar el aprendizaje de idiomas.',
         'translated_en':'Artificial intelligence can support language learning.',
         'detected_language':'es','inferred_location':'Madrid, Spain','latitude':'40.4168','longitude':'-3.7038',
         'location_confidence':1,'location_source':'synthetic demo','location_reason':'Fictional teaching example; not inferred from a real person.'},
        {'platform':'demo','username':'example_fr','date_iso':datetime.now().isoformat()+'Z',
         'original_text':'Les langues nous permettent de découvrir de nouvelles perspectives.',
         'translated_en':'Languages let us discover new perspectives.',
         'detected_language':'fr','inferred_location':'Paris, France','latitude':'48.8566','longitude':'2.3522',
         'location_confidence':1,'location_source':'synthetic demo','location_reason':'Fictional teaching example; not inferred from a real person.'}
    ]))
    print('Loaded two clearly labeled synthetic examples. No social or LLM API was called.')
    preview()


def require_data():
    if _sugar_df is None or _sugar_df.empty: raise ValueError('Load or collect data first.')


async def do_enrich():
    global _sugar_df
    require_data();validate_settings(translate.value or infer.value)
    if not (translate.value or infer.value or geocode.value): raise ValueError('Choose translation, location inference, or geocoding first.')
    # Retain existing columns/coordinates; only update the requested enrichment fields.
    df=normalise_frame(_sugar_df)
    client=create_llm_client() if (translate.value or infer.value) else None
    cache=load_json_cache(GEOCODE_CACHE_FILE)
    for index,row in df.iterrows():
        print(f'Processing row {index+1}/{len(df)}')
        language=row['detected_language'] or detect_language_safe(row['original_text'])
        df.at[index,'detected_language']=language
        if translate.value:
            df.at[index,'translated_en']=await translate_with_openai(client,row['original_text'],language,LLM_MODEL,target.value)
        if infer.value:
            location=await infer_location_with_openai(client,df.loc[index].to_dict(),LLM_MODEL)
            for col,key in [('inferred_location','location_name'),('location_confidence','confidence'),('location_source','source'),('location_reason','reason')]:
                df.at[index,col]=location[key]
            for col in ['latitude','longitude','geocode_display_name']: df.at[index,col]=''
        try: eligible=float(df.at[index,'location_confidence'] or 0)>=MIN_LOCATION_CONFIDENCE
        except (ValueError,TypeError): eligible=False
        if geocode.value and eligible and df.at[index,'inferred_location']:
            geo=await geocode_location(str(df.at[index,'inferred_location']),cache)
            for col,value in geo.items(): df.at[index,col]=value
        await asyncio.sleep(TRANSLATION_DELAY_SECONDS)
    _sugar_df=df;preview()


async def do_export():
    require_data()
    path=OUTPUT_DIR/f'social_search_posts_{timestamp()}.csv'
    # Export the whole frame, including extra columns from uploaded datasets.
    safe=_sugar_df.copy()
    numeric={'latitude','longitude','location_confidence'}
    for col in safe.columns:
        safe[col]=safe[col].map(lambda value: clean_cell(value,prevent_formula_injection=col not in numeric))
    # Numeric columns cannot carry spreadsheet formulas, even in untrusted uploads.
    for col in numeric & set(safe.columns): safe[col]=pd.to_numeric(safe[col],errors='coerce')
    safe.to_csv(path,index=False,encoding='utf-8-sig',quoting=csv.QUOTE_ALL)
    with pd.ExcelWriter(path.with_suffix('.xlsx'),engine='openpyxl') as writer:
        safe.to_excel(writer,index=False,sheet_name='posts')
        sheet=writer.sheets['posts'];sheet.freeze_panes='A2';sheet.auto_filter.ref=sheet.dimensions
        for cell in sheet[1]: cell.font=Font(bold=True);cell.fill=PatternFill('solid',fgColor='D9EAF7')
        for i,col in enumerate(safe.columns,1): sheet.column_dimensions[get_column_letter(i)].width=60 if col in {'original_text','translated_en'} else 25
        for row in sheet:
            for cell in row: cell.alignment=Alignment(vertical='top',wrap_text=True)
    offer_files([path,path.with_suffix('.xlsx')])


async def do_map():
    require_data()
    path=OUTPUT_DIR/f'social_search_map_{timestamp()}.html'
    map_object=create_tweet_map(normalise_frame(_sugar_df),str(path))
    if map_object is None: raise ValueError('No usable coordinates. Load latitude/longitude data or enable location inference and geocoding.')
    with results_view:
        results_view.clear_output(wait=True);display(map_object)
    offer_files([path])


def report_parts(df):
    parts=[('SUGAR data summary',f'{len(df)} rows. Generated {datetime.now().isoformat(timespec="seconds")}.')]
    parts.append(('Scope','Descriptive summary of the loaded dataset, not a representative population estimate. Language detection and inferred locations may be inaccurate.'))
    for col,label in [('platform','Sources'),('detected_language','Detected languages'),('query','Queries'),('inferred_location','Reported / inferred locations')]:
        values=df[col].astype(str).str.strip();counts=values[values.ne('')].value_counts().head(20)
        parts.append((label,'\n'.join(f'{name}: {count}' for name,count in counts.items()) or 'No values recorded.'))
    dates=pd.to_datetime(df['date_iso'],errors='coerce',utc=True).dropna()
    parts.append(('Date range',f'{dates.min()} to {dates.max()}' if len(dates) else 'No parseable dates.'))
    parts.append(('Sample posts','First 20 rows, in dataset order. Samples are not a random selection.'))
    for i,row in df.head(20).iterrows():
        parts.append((f"{i+1}. {row['platform']} · {row['username']}",str(row['original_text'])+'\n\nTranslation: '+str(row['translated_en'])))
    return parts


async def do_report():
    require_data()
    parts=report_parts(normalise_frame(_sugar_df))
    stem=OUTPUT_DIR/f'sugar_analysis_{timestamp()}';paths=[]
    if report_format.value in {'html','both'}:
        sections='\n'.join('<section><h2>'+html_lib.escape(title)+'</h2><p>'+html_lib.escape(text).replace('\n','<br>')+'</p></section>' for title,text in parts)
        html='<!doctype html><html lang="en"><meta charset="utf-8"><title>SUGAR data summary</title><style>body{font:16px/1.6 system-ui,sans-serif;max-width:850px;margin:40px auto;padding:24px;color:#202430}h2{font-size:20px}p{overflow-wrap:anywhere}section{break-inside:avoid}@media print{button,.print-help{display:none}body{margin:0;max-width:none}}</style><button onclick="window.print()">Print / Save as PDF</button><p class="print-help">Open this downloaded report in your browser and choose Print → Save as PDF. Your browser handles multilingual fonts.</p>'+sections+'</html>'
        path=stem.with_suffix('.html');path.write_text(html,encoding='utf-8');paths.append(path)
    if report_format.value in {'docx','both'}:
        from docx import Document
        from docx.shared import Pt
        document=Document();document.styles['Normal'].font.size=Pt(11)
        for title,text in parts:
            document.add_heading(title,level=1 if title==parts[0][0] else 2)
            document.add_paragraph(text)
        path=stem.with_suffix('.docx');document.save(path);paths.append(path)
    offer_files(paths)
    print('Report ready. For PDF, download the HTML report, open it, then Print → Save as PDF.')


async def run_action(action):
    global LLM_API_KEY
    for control in action_buttons + input_controls: control.disabled=True
    cancel_button.disabled=False
    set_status('Working…');NETWORK_FAILURES.clear()
    try:
        with log:
            log.clear_output(wait=True)
            await action()
        set_status('Finished. Some requests failed; see the log.' if NETWORK_FAILURES else 'Finished.')
    except asyncio.CancelledError:
        set_status('Cancelled. Existing files and previously loaded data are still available.')
    except Exception as exc:
        set_status('Could not complete: '+str(exc))
        with log: print(type(exc).__name__+': '+str(exc))
    finally:
        LLM_API_KEY=''
        for control in secret_fields: control.value=''
        for control in action_buttons + input_controls: control.disabled=False
        cancel_button.disabled=True


def start(action):
    global _sugar_task
    if _sugar_task is not None and not _sugar_task.done(): return
    _sugar_task=asyncio.create_task(run_action(action))


def cancel(_):
    if _sugar_task is not None and not _sugar_task.done(): _sugar_task.cancel()

for button,action in [(search_button,do_search),(load_button,do_load),(demo_button,do_demo),
                      (enrich_button,do_enrich),(export_button,do_export),(map_button,do_map),(report_button,do_report)]:
    button.on_click(lambda _, action=action: start(action))
cancel_button.on_click(cancel)

def provider_changed(change):
    model.value='gpt-oss-120b' if change['new']=='arc' else ''
    model.placeholder='Enter an available model ID for your API account'
provider.observe(provider_changed,names='value')

network_note=widgets.HTML('<p>Live APIs must allow browser requests (CORS). Use your VPN if ARC requires it. X access may require a server connector. Keys are cleared after each action; re-enter them when needed.</p>')
search_panel=widgets.VBox([terms,sources,query_languages,widgets.HBox([since,until]),widgets.HBox([post_limit,page_limit]),retweets,x_mode,x_languages,handles,search_button])
credentials_panel=widgets.VBox([network_note,x_key,bsky_handle,bsky_password,masto_instance,masto_key])
llm_panel=widgets.VBox([provider,model,llm_key,translate,target,infer,confidence,geocode,geocode_endpoint,widgets.HTML('<p>Location estimates concern broad public places, not private addresses. The public Nominatim service allows at most one request/second across the entire application, not per student. This notebook paces one browser, but cannot coordinate a classroom. For class-wide use, provide a geocoder with shared throttling or pre-geocoded data. <a href="https://operations.osmfoundation.org/policies/nominatim/" target="_blank">Usage policy</a>. Existing coordinates can be mapped without geocoding.</p>')])
accordion=widgets.Accordion(children=[search_panel,credentials_panel,llm_panel],selected_index=0)
for i,title in enumerate(['Search settings','Social API credentials','Translation and location settings']):accordion.set_title(i,title)
_sugar_ui=widgets.VBox([widgets.HTML('<h2>SUGAR · Browser edition</h2><p>Search, translate, map, and export. Try the synthetic demo without API credentials.</p>'),
    widgets.HBox([demo_button,cancel_button]),status,accordion,
    widgets.HTML('<h3>Use saved data</h3>'),upload,file_path,load_button,
    widgets.HTML('<h3>Work with loaded results</h3>'),enrich_button,widgets.HBox([export_button,map_button]),report_format,report_button,
    widgets.HTML('<h3>Progress</h3>'),log,widgets.HTML('<h3>Results</h3>'),results_view,widgets.HTML('<h3>Downloads</h3>'),downloads_view],layout=FULL)
display(_sugar_ui)
